# Apache Spark (Local) vs Pandas
### *From Single-Machine to Distributed Thinking*



---
## 1. 🏗️ Setup & Architecture

### How Spark works

Spark distributes data into **partitions** that are processed in parallel across multiple nodes.  
In **local mode**, all of this happens on your single machine — perfect for learning.

```
┌──────────────────────────────────────────────────────────────┐
│                       SPARK CLUSTER                          │
│                                                              │
│   ┌───────────┐     ┌────────────┐     ┌────────────┐        │
│   │  Driver   │────▶│  Executor  │     │  Executor  │        │
│   │ (your code│     │ (Worker 1) │     │ (Worker 2) │        │
│   │  & plans) │     │ Partition1 │     │ Partition2 │        │
│   └───────────┘     └────────────┘     └────────────┘        │
│                                                              │
│         local[*] mode → everything runs on ONE machine       │
└──────────────────────────────────────────────────────────────┘
```

### Pandas vs Spark at a glance

| | Pandas | Spark |
|-|--------|-------|
| **Runs on** | 1 machine, 1 process | Cluster of machines (or local) |
| **Data lives** | In RAM | Distributed across partitions |
| **Execution** | Immediate (eager) | Lazy — runs only on an action |
| **Scale** | Up to a few GBs | Terabytes / Petabytes |
| **Entry point** | `import pandas as pd` | `SparkSession` |

In [ ]:
# Install if needed
# !pip install pyspark

In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F   # Spark's built-in function library
from pyspark.sql.types import *          # schema type definitions
from pyspark.sql.window import Window    # window function specs

import pyspark
print(f"Pandas  : {pd.__version__}")
print(f"PySpark : {pyspark.__version__}")

Pandas  : 2.2.3
PySpark : 4.0.4


In [ ]:
# Creating the SparkSession — the single entry point for all Spark functionality.
#   local[*]  → use ALL available CPU cores
#   local[2]  → use exactly 2 cores

spark = (
    SparkSession.builder
    .appName("Spark vs Pandas - Class")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")         # small dataset, fewer partitions
    .config("spark.sql.repl.eagerEval.enabled", True)    # pretty display in notebooks
    .getOrCreate()
)

spark

In [ ]:
# While cells are running, open the Spark UI in your browser
print(f"Spark UI : http://localhost:4040")
print(f"Version  : {spark.version}")
print(f"App Name : {spark.sparkContext.appName}")
print(f"Master   : {spark.sparkContext.master}")

Spark UI : http://localhost:4040
Version  : 4.0.4
App Name : Spark vs Pandas - Class
Master   : local[*]


---
## 2. 📊 Creating DataFrames

> We'll use an e-commerce sales dataset throughout the entire notebook.

In [ ]:
# Shared raw data used by both Pandas and Spark
data = {
    "order_id":  [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010,
                  1011, 1012, 1013, 1014, 1015],
    "product":   ["Notebook", "Mouse", "Keyboard", "Monitor", "Headset",
                  "Webcam", "External HD", "Pendrive", "USB Cable", "USB Hub",
                  "Notebook", "Monitor", "Keyboard", "Mouse", "Headset"],
    "category":  ["Computers", "Peripherals", "Peripherals", "Computers", "Audio",
                  "Peripherals", "Storage", "Storage", "Cables", "Peripherals",
                  "Computers", "Computers", "Peripherals", "Peripherals", "Audio"],
    "price":     [3500, 85, 150, 1200, 250, 320, 400, 60, 25, 120,
                  3200, 1100, 145, 90, 280],
    "quantity":  [1, 3, 2, 1, 2, 1, 1, 5, 10, 2,
                  2, 1, 3, 2, 1],
    "rating":    [4.8, 4.2, 4.5, 4.7, 3.9, 4.1, 4.6, 4.0, None, 4.3,
                  4.9, 4.6, 4.4, 4.1, 4.0],
    "region":    ["South", "North", "Southeast", "Southeast", "North",
                  "South", "Southeast", "North", "South", "Southeast",
                  "North", "South", "Southeast", "North", "Southeast"],
    "sale_date": ["2024-01-15", "2024-01-18", "2024-02-03", "2024-02-10", "2024-02-14",
                  "2024-03-01", "2024-03-15", "2024-03-22", "2024-04-05", "2024-04-18",
                  "2024-05-02", "2024-05-20", "2024-06-08", "2024-06-15", "2024-06-30"]
}

### 2.1 From a Python dictionary

In [ ]:
# ════════════════════════════════
#  🐼 PANDAS
# ════════════════════════════════
pdf = pd.DataFrame(data)
pdf.head(5)

,order_id,product,category,price,quantity,rating,region,sale_date
0,1001,Notebook,Computers,3500,1,4.8,South,2024-01-15
1,1002,Mouse,Peripherals,85,3,4.2,North,2024-01-18
2,1003,Keyboard,Peripherals,150,2,4.5,Southeast,2024-02-03
3,1004,Monitor,Computers,1200,1,4.7,Southeast,2024-02-10
4,1005,Headset,Audio,250,2,3.9,North,2024-02-14


In [ ]:
# ════════════════════════════════
#  🔥 SPARK
# ════════════════════════════════
# Most common in dev/learning: convert from a Pandas DataFrame
sdf = spark.createDataFrame(pdf)
sdf.show(5, truncate=False)

+--------+--------+-----------+-----+--------+------+---------+----------+
|order_id|product |category   |price|quantity|rating|region   |sale_date |
+--------+--------+-----------+-----+--------+------+---------+----------+
|1001    |Notebook|Computers  |3500 |1       |4.8   |South    |2024-01-15|
|1002    |Mouse   |Peripherals|85   |3       |4.2   |North    |2024-01-18|
|1003    |Keyboard|Peripherals|150  |2       |4.5   |Southeast|2024-02-03|
|1004    |Monitor |Computers  |1200 |1       |4.7   |Southeast|2024-02-10|
|1005    |Headset |Audio      |250  |2       |3.9   |North    |2024-02-14|
+--------+--------+-----------+-----+--------+------+---------+----------+
only showing top 5 rows


In [ ]:
# With an explicit schema — recommended in production.
# Avoids schema inference costs and makes intent clear.

schema = StructType([
    StructField("order_id",  IntegerType(), nullable=False),
    StructField("product",   StringType(),  nullable=False),
    StructField("category",  StringType(),  nullable=True),
    StructField("price",     IntegerType(), nullable=False),
    StructField("quantity",  IntegerType(), nullable=False),
    StructField("rating",    DoubleType(),  nullable=True),
    StructField("region",    StringType(),  nullable=True),
    StructField("sale_date", StringType(),  nullable=True),
])

rows = list(zip(*[data[k] for k in schema.fieldNames()]))
sdf_typed = spark.createDataFrame(rows, schema=schema) # criando a tabela virtual
sdf_typed.printSchema()

root
 |-- order_id: integer (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- rating: double (nullable = true)
 |-- region: string (nullable = true)
 |-- sale_date: string (nullable = true)



### 2.2 Reading files

In [ ]:
# 🐼 Pandas readers
# pdf = pd.read_csv("sales.csv")
# pdf = pd.read_parquet("sales.parquet")
# pdf = pd.read_json("sales.json")

# vector: list[list] = [[1,2,3,4], [0.7,4,5,3,3]]
# read_csv, "string", ast.literal_eval

# 🔥 Spark readers
# sdf = spark.read.csv("sales.csv", header=True, inferSchema=True)
# sdf = spark.read.parquet("sales.parquet")     # ← preferred columnar format
# sdf = spark.read.json("sales.json")
# sdf = spark.read.format("delta").load("path") # ← Delta Lake

# Round-trip through Parquet
pdf.to_parquet("/tmp/sales.parquet", index=False)
sdf_parquet = spark.read.parquet("/tmp/sales.parquet")

print("Read from Parquet:")
sdf_parquet.show(3, truncate=False)

Read from Parquet:
+--------+--------+-----------+-----+--------+------+---------+----------+
|order_id|product |category   |price|quantity|rating|region   |sale_date |
+--------+--------+-----------+-----+--------+------+---------+----------+
|1001    |Notebook|Computers  |3500 |1       |4.8   |South    |2024-01-15|
|1002    |Mouse   |Peripherals|85   |3       |4.2   |North    |2024-01-18|
|1003    |Keyboard|Peripherals|150  |2       |4.5   |Southeast|2024-02-03|
+--------+--------+-----------+-----+--------+------+---------+----------+
only showing top 3 rows


### 2.3 Converting between Pandas and Spark

> ⚠️ **`toPandas()`** collects **all data** to the driver node.  
> Only use this on small DataFrames — it defeats the purpose of distributed processing on large data.

In [ ]:
# Spark → Pandas  (collects everything to the driver)
pdf_from_spark = sdf.toPandas()
print("Spark → Pandas:", type(pdf_from_spark))

# Pandas → Spark
sdf_from_pandas = spark.createDataFrame(pdf)
print("Pandas → Spark:", type(sdf_from_pandas))

Spark → Pandas: <class 'pandas.core.frame.DataFrame'>
Pandas → Spark: <class 'pyspark.sql.classic.dataframe.DataFrame'>


In [ ]:
pdf["product"].value_counts()

,count
product,
Notebook,2
Mouse,2
Keyboard,2
Monitor,2
Headset,2
Webcam,1
External HD,1
Pendrive,1
USB Cable,1


---
## 3. 🔍 Exploration

| Operation | Pandas | Spark |
|-----------|--------|-------|
| Dimensions | `df.shape` | `df.count(), len(df.columns)` |
| First rows | `df.head(n)` | `df.show(n)` |
| Column types | `df.dtypes` | `df.printSchema()` |
| Statistics | `df.describe()` | `df.describe()` |
| Null counts | `df.isnull().sum()` | `df.select([F.count(F.when(F.isnull(c), c))...])` |
| Unique values | `df["col"].unique()` | `df.select("col").distinct()` |
| Value counts | `df["col"].value_counts()` | `df.groupBy("col").count()` |

In [ ]:
# Shape / Dimensions
print(f"🐼 Pandas  shape : {pdf.shape}")

# ⚠️ .count() triggers execution in Spark
# print(f"🔥 Spark   rows  : {sdf.count()}  |  cols: {len(sdf.columns)}")

🐼 Pandas  shape : (15, 8)


In [ ]:
# Schema / Types
print("🐼 Pandas dtypes:")
print(pdf.dtypes)
print()
print("🔥 Spark schema:")
sdf.printSchema()

🐼 Pandas dtypes:
order_id       int64
product       object
category      object
price          int64
quantity       int64
rating       float64
region        object
sale_date     object
dtype: object

🔥 Spark schema:
root
 |-- order_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- rating: double (nullable = true)
 |-- region: string (nullable = true)
 |-- sale_date: string (nullable = true)



In [ ]:
# First rows
print("🐼 Pandas .head():")
display(pdf.head(4))

print("\n🔥 Spark .show():")
sdf.show(4, truncate=False)

🐼 Pandas .head():


,order_id,product,category,price,quantity,rating,region,sale_date
0,1001,Notebook,Computers,3500,1,4.8,South,2024-01-15
1,1002,Mouse,Peripherals,85,3,4.2,North,2024-01-18
2,1003,Keyboard,Peripherals,150,2,4.5,Southeast,2024-02-03
3,1004,Monitor,Computers,1200,1,4.7,Southeast,2024-02-10



🔥 Spark .show():
+--------+--------+-----------+-----+--------+------+---------+----------+
|order_id|product |category   |price|quantity|rating|region   |sale_date |
+--------+--------+-----------+-----+--------+------+---------+----------+
|1001    |Notebook|Computers  |3500 |1       |4.8   |South    |2024-01-15|
|1002    |Mouse   |Peripherals|85   |3       |4.2   |North    |2024-01-18|
|1003    |Keyboard|Peripherals|150  |2       |4.5   |Southeast|2024-02-03|
|1004    |Monitor |Computers  |1200 |1       |4.7   |Southeast|2024-02-10|
+--------+--------+-----------+-----+--------+------+---------+----------+
only showing top 4 rows


In [ ]:
# Descriptive statistics
print("🐼 Pandas .describe():")
display(pdf.describe())

print("\n🔥 Spark .describe():")
sdf.describe("price", "quantity", "rating").show()

🐼 Pandas .describe():


,order_id,price,quantity,rating
count,15.000000,15.000000,15.000000,14.000000
mean,1008.000000,728.333333,2.466667,4.364286
std,4.472136,1123.252081,2.356349,0.324884
min,1001.000000,25.000000,1.000000,3.900000
25%,1004.500000,105.000000,1.000000,4.100000
50%,1008.000000,250.000000,2.000000,4.350000
75%,1011.500000,750.000000,2.500000,4.600000
max,1015.000000,3500.000000,10.000000,4.900000



🔥 Spark .describe():
+-------+------------------+-----------------+------+
|summary|             price|         quantity|rating|
+-------+------------------+-----------------+------+
|  count|                15|               15|    15|
|   mean| 728.3333333333334|2.466666666666667|   NaN|
| stddev|1123.2520812779464|2.356349072692956|   NaN|
|    min|                25|                1|   3.9|
|    max|              3500|               10|   NaN|
+-------+------------------+-----------------+------+



In [ ]:
# Null counts per column
print("🐼 Pandas isnull().sum():")
print(pdf.isnull().sum())

print("\n🔥 Spark null counts:")
sdf.select([
    F.count(F.when(F.isnull(c), c)).alias(c)
    for c in sdf.columns
]).show()

🐼 Pandas isnull().sum():
order_id     0
product      0
category     0
price        0
quantity     0
rating       1
region       0
sale_date    0
dtype: int64

🔥 Spark null counts:
+--------+-------+--------+-----+--------+------+------+---------+
|order_id|product|category|price|quantity|rating|region|sale_date|
+--------+-------+--------+-----+--------+------+------+---------+
|       0|      0|       0|    0|       0|     0|     0|        0|
+--------+-------+--------+-----+--------+------+------+---------+



In [ ]:
# Unique values and value counts
print("🐼 Pandas unique():")
print(pdf["category"].unique())
print()
print("🐼 Pandas value_counts():")
print(pdf["category"].value_counts())

print("\n🔥 Spark .distinct():")
sdf.select("category").distinct().show()

print("\n🔥 Spark groupBy().count() — equivalent to value_counts():")
sdf.groupBy("category").count().orderBy("count", ascending=False).show()

🐼 Pandas unique():
['Computers' 'Peripherals' 'Audio' 'Storage' 'Cables']

🐼 Pandas value_counts():
category
Peripherals    6
Computers      4
Audio          2
Storage        2
Cables         1
Name: count, dtype: int64

🔥 Spark .distinct():
+-----------+
|   category|
+-----------+
|  Computers|
|Peripherals|
|      Audio|
|    Storage|
|     Cables|
+-----------+


🔥 Spark groupBy().count() — equivalent to value_counts():
+-----------+-----+
|   category|count|
+-----------+-----+
|Peripherals|    6|
|  Computers|    4|
|      Audio|    2|
|    Storage|    2|
|     Cables|    1|
+-----------+-----+



---
## 4. 🎯 Selection & Filtering

| Operation | Pandas | Spark |
|-----------|--------|-------|
| Select columns | `df[["col1", "col2"]]` | `df.select("col1", "col2")` |
| Filter rows | `df[df["col"] > x]` | `df.filter(df.col > x)` |
| SQL-style filter | `df.query("col > x")` | `df.filter("col > x")` |
| IN operator | `df[df["col"].isin([...])]` | `df.filter(df.col.isin([...]))` |
| Null filter | `df[df["col"].isnull()]` | `df.filter(F.col("col").isNull())` |
| Sort | `df.sort_values("col")` | `df.orderBy("col")` |

In [ ]:
# Selecting columns
print("🐼 Pandas:")
display(pdf[["order_id", "product", "price", "quantity"]].head(5))

print("\n🔥 Spark:")
sdf.select("order_id", "product", "price", "quantity").show(5)

🐼 Pandas:


,order_id,product,price,quantity
0,1001,Notebook,3500,1
1,1002,Mouse,85,3
2,1003,Keyboard,150,2
3,1004,Monitor,1200,1
4,1005,Headset,250,2



🔥 Spark:
+--------+--------+-----+--------+
|order_id| product|price|quantity|
+--------+--------+-----+--------+
|    1001|Notebook| 3500|       1|
|    1002|   Mouse|   85|       3|
|    1003|Keyboard|  150|       2|
|    1004| Monitor| 1200|       1|
|    1005| Headset|  250|       2|
+--------+--------+-----+--------+
only showing top 5 rows


In [ ]:
# Simple filter — Spark has two equivalent syntaxes
print("🐼 Pandas — price > 500:")
display(pdf[pdf["price"] > 500][["product", "price", "category"]])

print("\n🔥 Spark — column syntax:")
sdf.filter(sdf.price > 500).select("product", "price", "category").show()

print("\n🔥 Spark — SQL string syntax (mirrors Pandas .query()):")
sdf.filter("price > 500").select("product", "price", "category").show()

🐼 Pandas — price > 500:


,product,price,category
0,Notebook,3500,Computers
3,Monitor,1200,Computers
10,Notebook,3200,Computers
11,Monitor,1100,Computers



🔥 Spark — column syntax:
+--------+-----+---------+
| product|price| category|
+--------+-----+---------+
|Notebook| 3500|Computers|
| Monitor| 1200|Computers|
|Notebook| 3200|Computers|
| Monitor| 1100|Computers|
+--------+-----+---------+


🔥 Spark — SQL string syntax (mirrors Pandas .query()):
+--------+-----+---------+
| product|price| category|
+--------+-----+---------+
|Notebook| 3500|Computers|
| Monitor| 1200|Computers|
|Notebook| 3200|Computers|
| Monitor| 1100|Computers|
+--------+-----+---------+



In [ ]:
# Multiple conditions:  & = AND   | = OR   ~ = NOT
print("🐼 Pandas — price > 100 AND rating >= 4.5:")
mask = (pdf["price"] > 100) & (pdf["rating"] >= 4.5)
display(pdf[mask][["product", "price", "rating", "category"]])

print("\n🔥 Spark — same logic:")
sdf.filter(
    (F.col("price") > 100) & (F.col("rating") >= 4.5)
).select("product", "price", "rating", "category").show()

🐼 Pandas — price > 100 AND rating >= 4.5:


,product,price,rating,category
0,Notebook,3500,4.8,Computers
2,Keyboard,150,4.5,Peripherals
3,Monitor,1200,4.7,Computers
6,External HD,400,4.6,Storage
10,Notebook,3200,4.9,Computers
11,Monitor,1100,4.6,Computers



🔥 Spark — same logic:
+-----------+-----+------+-----------+
|    product|price|rating|   category|
+-----------+-----+------+-----------+
|   Notebook| 3500|   4.8|  Computers|
|   Keyboard|  150|   4.5|Peripherals|
|    Monitor| 1200|   4.7|  Computers|
|External HD|  400|   4.6|    Storage|
|   Notebook| 3200|   4.9|  Computers|
|    Monitor| 1100|   4.6|  Computers|
+-----------+-----+------+-----------+



In [ ]:
# isin — IN operator
targets = ["Audio", "Storage"]

# (pdf["category"] == "Audio") | (pdf["category"] == "Storage")

print("🐼 Pandas .isin():")
display(pdf[pdf["category"].isin(targets)][["product", "category", "price"]])

print("\n🔥 Spark .isin():")
sdf.filter(F.col("category").isin(targets)).select("product", "category", "price").show()

🐼 Pandas .isin():


,product,category,price
4,Headset,Audio,250
6,External HD,Storage,400
7,Pendrive,Storage,60
14,Headset,Audio,280



🔥 Spark .isin():
+-----------+--------+-----+
|    product|category|price|
+-----------+--------+-----+
|    Headset|   Audio|  250|
|External HD| Storage|  400|
|   Pendrive| Storage|   60|
|    Headset|   Audio|  280|
+-----------+--------+-----+



In [ ]:
# Filtering nulls
print("🐼 Pandas — rows where rating is null:")
display(pdf[pdf["rating"].isnull()])

print("\n🔥 Spark — rows where rating is null:")
sdf.filter(F.col("rating").isNull()).show()

🐼 Pandas — rows where rating is null:


,order_id,product,category,price,quantity,rating,region,sale_date
8,1009,USB Cable,Cables,25,10,NaN,South,2024-04-05



🔥 Spark — rows where rating is null:
+--------+-------+--------+-----+--------+------+------+---------+
|order_id|product|category|price|quantity|rating|region|sale_date|
+--------+-------+--------+-----+--------+------+------+---------+
+--------+-------+--------+-----+--------+------+------+---------+



In [ ]:
# Sorting by multiple columns
print("🐼 Pandas .sort_values():")
display(
    pdf.sort_values(["price", "rating"], ascending=[False, False])
    [["product", "price", "rating"]].head(5)
)

print("\n🔥 Spark .orderBy():")
sdf.orderBy(
    F.col("price").desc(), F.col("rating").desc()
).select("product", "price", "rating").show(5)

🐼 Pandas .sort_values():


,product,price,rating
0,Notebook,3500,4.8
10,Notebook,3200,4.9
3,Monitor,1200,4.7
11,Monitor,1100,4.6
6,External HD,400,4.6



🔥 Spark .orderBy():
+-----------+-----+------+
|    product|price|rating|
+-----------+-----+------+
|   Notebook| 3500|   4.8|
|   Notebook| 3200|   4.9|
|    Monitor| 1200|   4.7|
|    Monitor| 1100|   4.6|
|External HD|  400|   4.6|
+-----------+-----+------+
only showing top 5 rows


---
## 5. 🔧 Transformations

| Operation | Pandas | Spark |
|-----------|--------|-------|
| Add/modify column | `df["new"] = expr` | `df.withColumn("new", expr)` |
| Drop column | `df.drop(columns=["c"])` | `df.drop("c")` |
| Rename column | `df.rename(columns={"a": "b"})` | `df.withColumnRenamed("a", "b")` |
| Conditional value | `np.where(cond, a, b)` | `F.when(cond, a).otherwise(b)` |
| Fill nulls | `df["c"].fillna(v)` | `df.fillna({"c": v})` |
| Drop null rows | `df.dropna()` | `df.dropna()` |
| Custom function | `df["c"].apply(fn)` | `F.udf(fn)(col)` |

In [ ]:
# Creating new columns
pdf2 = pdf.copy()
pdf2["total"]        = pdf2["price"] * pdf2["quantity"]
pdf2["discounted"]   = (pdf2["price"] * 0.90).round(2)
pdf2["is_expensive"] = pdf2["price"] > 500

print("🐼 Pandas — new columns:")
display(pdf2[["product", "price", "quantity", "total", "discounted", "is_expensive"]].head(5))

sdf2 = (
    sdf
    .withColumn("total",        F.col("price") * F.col("quantity"))
    .withColumn("discounted",   F.round(F.col("price") * 0.90, 2))
    .withColumn("is_expensive", F.col("price") > 500)
)
print("\n🔥 Spark — withColumn:")
sdf2.select("product", "price", "quantity", "total", "discounted", "is_expensive").show(5)

🐼 Pandas — new columns:


,product,price,quantity,total,discounted,is_expensive
0,Notebook,3500,1,3500,3150.0,True
1,Mouse,85,3,255,76.5,False
2,Keyboard,150,2,300,135.0,False
3,Monitor,1200,1,1200,1080.0,True
4,Headset,250,2,500,225.0,False



🔥 Spark — withColumn:
+--------+-----+--------+-----+----------+------------+
| product|price|quantity|total|discounted|is_expensive|
+--------+-----+--------+-----+----------+------------+
|Notebook| 3500|       1| 3500|    3150.0|        true|
|   Mouse|   85|       3|  255|      76.5|       false|
|Keyboard|  150|       2|  300|     135.0|       false|
| Monitor| 1200|       1| 1200|    1080.0|        true|
| Headset|  250|       2|  500|     225.0|       false|
+--------+-----+--------+-----+----------+------------+
only showing top 5 rows


In [ ]:
# Conditional column
# Pandas: np.select()    |    Spark: F.when().when().otherwise()

pdf2["price_range"] = np.select(
    [pdf2["price"] < 100, pdf2["price"] < 500],
    ["Budget",            "Mid-range"],
    default="Premium"
)
print("🐼 Pandas np.select():")
display(pdf2[["product", "price", "price_range"]].drop_duplicates(subset=["price_range"]).sort_values("price"))

sdf2 = sdf2.withColumn(
    "price_range",
    F.when(F.col("price") < 100, "Budget")
     .when(F.col("price") < 500, "Mid-range")
     .otherwise("Premium")
)
print("\n🔥 Spark F.when().otherwise():")
sdf2.select("product", "price", "price_range").show()

🐼 Pandas np.select():


,product,price,price_range
1,Mouse,85,Budget
2,Keyboard,150,Mid-range
0,Notebook,3500,Premium



🔥 Spark F.when().otherwise():
+-----------+-----+-----------+
|    product|price|price_range|
+-----------+-----+-----------+
|   Notebook| 3500|    Premium|
|      Mouse|   85|     Budget|
|   Keyboard|  150|  Mid-range|
|    Monitor| 1200|    Premium|
|    Headset|  250|  Mid-range|
|     Webcam|  320|  Mid-range|
|External HD|  400|  Mid-range|
|   Pendrive|   60|     Budget|
|  USB Cable|   25|     Budget|
|    USB Hub|  120|  Mid-range|
|   Notebook| 3200|    Premium|
|    Monitor| 1100|    Premium|
|   Keyboard|  145|  Mid-range|
|      Mouse|   90|     Budget|
|    Headset|  280|  Mid-range|
+-----------+-----+-----------+



In [ ]:
# String operations
# Pandas uses the .str accessor; Spark has F.upper(), F.length(), F.concat(), etc.

pdf2["product_upper"] = pdf2["product"].str.upper()
pdf2["product_len"]   = pdf2["product"].str.len()
pdf2["label"]         = pdf2["product"] + " — " + pdf2["category"]

print("🐼 Pandas string ops:")
display(pdf2[["product", "product_upper", "product_len", "label"]].head(5))

sdf2 = (
    sdf2
    .withColumn("product_upper", F.upper(F.col("product")))
    .withColumn("product_len",   F.length(F.col("product")))
    .withColumn("label",         F.concat(F.col("product"), F.lit(" — "), F.col("category")))
)
print("\n🔥 Spark string functions:")
sdf2.select("product", "product_upper", "product_len", "label").show(5, truncate=False)

🐼 Pandas string ops:


,product,product_upper,product_len,label
0,Notebook,NOTEBOOK,8,Notebook — Computers
1,Mouse,MOUSE,5,Mouse — Peripherals
2,Keyboard,KEYBOARD,8,Keyboard — Peripherals
3,Monitor,MONITOR,7,Monitor — Computers
4,Headset,HEADSET,7,Headset — Audio



🔥 Spark string functions:
+--------+-------------+-----------+----------------------+
|product |product_upper|product_len|label                 |
+--------+-------------+-----------+----------------------+
|Notebook|NOTEBOOK     |8          |Notebook — Computers  |
|Mouse   |MOUSE        |5          |Mouse — Peripherals   |
|Keyboard|KEYBOARD     |8          |Keyboard — Peripherals|
|Monitor |MONITOR      |7          |Monitor — Computers   |
|Headset |HEADSET      |7          |Headset — Audio       |
+--------+-------------+-----------+----------------------+
only showing top 5 rows


In [ ]:
# Date operations
pdf2["sale_date"]  = pd.to_datetime(pdf2["sale_date"])
pdf2["year"]       = pdf2["sale_date"].dt.year
pdf2["month"]      = pdf2["sale_date"].dt.month
pdf2["month_name"] = pdf2["sale_date"].dt.strftime("%B")

print("🐼 Pandas date ops:")
display(pdf2[["sale_date", "year", "month", "month_name"]].head(5))

sdf2 = (
    sdf2
    .withColumn("sale_date",  F.to_date(F.col("sale_date"), "yyyy-MM-dd"))
    .withColumn("year",       F.year(F.col("sale_date")))
    .withColumn("month",      F.month(F.col("sale_date")))
    .withColumn("month_name", F.date_format(F.col("sale_date"), "MMMM"))
)
print("\n🔥 Spark date functions:")
sdf2.select("sale_date", "year", "month", "month_name").show(5)

🐼 Pandas date ops:


,sale_date,year,month,month_name
0,2024-01-15,2024,1,January
1,2024-01-18,2024,1,January
2,2024-02-03,2024,2,February
3,2024-02-10,2024,2,February
4,2024-02-14,2024,2,February



🔥 Spark date functions:
+----------+----+-----+----------+
| sale_date|year|month|month_name|
+----------+----+-----+----------+
|2024-01-15|2024|    1|   January|
|2024-01-18|2024|    1|   January|
|2024-02-03|2024|    2|  February|
|2024-02-10|2024|    2|  February|
|2024-02-14|2024|    2|  February|
+----------+----+-----+----------+
only showing top 5 rows


In [ ]:
# Handling nulls
avg_pd = pdf["rating"].mean()
pdf2["rating"] = pdf2["rating"].fillna(avg_pd)
print(f"🐼 Pandas — filled nulls with mean: {avg_pd:.3f}")
print(f"   Nulls remaining: {pdf2['rating'].isnull().sum()}")

avg_sp = sdf.select(F.avg("rating")).collect()[0][0]
sdf2 = sdf2.fillna({"rating": round(avg_sp, 3)})
print(f"\n🔥 Spark — filled nulls with mean: {avg_sp:.3f}")
print(f"   Nulls remaining: {sdf2.filter(F.col('rating').isNull()).count()}")

🐼 Pandas — filled nulls with mean: 4.364
   Nulls remaining: 0

🔥 Spark — filled nulls with mean: nan
   Nulls remaining: 0


### 5.1 UDFs — User Defined Functions

> **UDFs** let you use custom Python functions inside Spark transformations.  
> They are the Spark equivalent of Pandas `.apply()`.  
> ⚠️ UDFs are slower than native `F.*` functions because they break out of the JVM.  
> Always prefer built-in Spark functions when they exist.

In [ ]:
def classify_price(price):
    """Classifies a price as Budget / Mid-range / Premium."""
    if price is None:  return "Unknown"
    if price < 100:    return "Budget"
    if price < 500:    return "Mid-range"
    return "Premium"

# 🐼 Pandas — .apply()
pdf2["price_class"] = pdf2["price"].apply(classify_price)
print("🐼 Pandas .apply():")
display(pdf2[["product", "price", "price_class"]].head(5))

# 🔥 Spark — UDF
classify_udf = F.udf(classify_price, StringType())
sdf2 = sdf2.withColumn("price_class", classify_udf(F.col("price")))
print("\n🔥 Spark UDF:")
sdf2.select("product", "price", "price_class").show(5)

🐼 Pandas .apply():


,product,price,price_class
0,Notebook,3500,Premium
1,Mouse,85,Budget
2,Keyboard,150,Mid-range
3,Monitor,1200,Premium
4,Headset,250,Mid-range



🔥 Spark UDF:
+--------+-----+-----------+
| product|price|price_class|
+--------+-----+-----------+
|Notebook| 3500|    Premium|
|   Mouse|   85|     Budget|
|Keyboard|  150|  Mid-range|
| Monitor| 1200|    Premium|
| Headset|  250|  Mid-range|
+--------+-----+-----------+
only showing top 5 rows


In [ ]:
# Renaming and dropping columns
pdf3 = pdf2.rename(columns={"price": "price_usd", "rating": "score"})
pdf3 = pdf3.drop(columns=["product_upper", "product_len", "label"])
print("🐼 Pandas columns after rename/drop:")
print(pdf3.columns.tolist())

sdf3 = (
    sdf2
    .withColumnRenamed("price", "price_usd")
    .withColumnRenamed("rating", "score")
    .drop("product_upper", "product_len", "label")
)
print("\n🔥 Spark columns after rename/drop:")
print(sdf3.columns)

🐼 Pandas columns after rename/drop:
['order_id', 'product', 'category', 'price_usd', 'quantity', 'score', 'region', 'sale_date', 'total', 'discounted', 'is_expensive', 'price_range', 'year', 'month', 'month_name', 'price_class']

🔥 Spark columns after rename/drop:
['order_id', 'product', 'category', 'price_usd', 'quantity', 'score', 'region', 'sale_date', 'total', 'discounted', 'is_expensive', 'price_range', 'year', 'month', 'month_name', 'price_class']


---
## 6. 📊 GroupBy & Aggregations

| Function | Pandas | Spark |
|----------|--------|-------|
| Group + agg | `.groupby("c").agg(...)` | `.groupBy("c").agg(...)` |
| Sum | `("c", "sum")` | `F.sum("c")` |
| Average | `("c", "mean")` | `F.avg("c")` |
| Count | `("c", "count")` | `F.count("c")` |
| Min / Max | `("c", "min/max")` | `F.min("c") / F.max("c")` |
| Keep shape | `.transform("mean")` | `F.avg().over(Window(...))` |

In [ ]:
# Basic GroupBy with multiple aggregations
print("🐼 Pandas groupby + agg:")
pd_summary = (
    pdf2
    .groupby("category")
    .agg(
        orders        = ("order_id", "count"),
        total_revenue = ("total",    "sum"),
        avg_price     = ("price",    "mean"),
        avg_rating    = ("rating",   "mean"),
        max_price     = ("price",    "max")
    )
    .round(2)
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
display(pd_summary)

print("\n🔥 Spark groupBy + agg:")
(
    sdf2
    .groupBy("category")
    .agg(
        F.count("order_id")          .alias("orders"),
        F.sum("total")               .alias("total_revenue"),
        F.round(F.avg("price"), 2)   .alias("avg_price"),
        F.round(F.avg("rating"), 2)  .alias("avg_rating"),
        F.max("price")               .alias("max_price")
    )
    .orderBy(F.col("total_revenue").desc())
).show()

🐼 Pandas groupby + agg:


,category,orders,total_revenue,avg_price,avg_rating,max_price
2,Computers,4,12200,2250.00,4.75,3500
3,Peripherals,6,1730,151.67,4.27,320
0,Audio,2,780,265.00,3.95,280
4,Storage,2,700,230.00,4.30,400
1,Cables,1,250,25.00,4.36,25



🔥 Spark groupBy + agg:
+-----------+------+-------------+---------+----------+---------+
|   category|orders|total_revenue|avg_price|avg_rating|max_price|
+-----------+------+-------------+---------+----------+---------+
|  Computers|     4|        12200|   2250.0|      4.75|     3500|
|Peripherals|     6|         1730|   151.67|      4.27|      320|
|      Audio|     2|          780|    265.0|      3.95|      280|
|    Storage|     2|          700|    230.0|       4.3|      400|
|     Cables|     1|          250|     25.0|       NaN|       25|
+-----------+------+-------------+---------+----------+---------+



In [ ]:
# GroupBy on multiple columns
print("🐼 Pandas — category x region:")
display(
    pdf2
    .groupby(["category", "region"])
    .agg(orders=("order_id", "count"), revenue=("total", "sum"))
    .reset_index()
    .sort_values("revenue", ascending=False)
    .head(8)
)

print("\n🔥 Spark — category x region:")
sdf2.groupBy("category", "region").agg(
    F.count("order_id").alias("orders"),
    F.sum("total").alias("revenue")
).orderBy(F.col("revenue").desc()).show(8)

🐼 Pandas — category x region:


,category,region,orders,revenue
3,Computers,North,1,6400
4,Computers,South,2,4600
5,Computers,Southeast,1,1200
8,Peripherals,Southeast,3,975
0,Audio,North,1,500
6,Peripherals,North,2,435
10,Storage,Southeast,1,400
7,Peripherals,South,1,320



🔥 Spark — category x region:
+-----------+---------+------+-------+
|   category|   region|orders|revenue|
+-----------+---------+------+-------+
|  Computers|    North|     1|   6400|
|  Computers|    South|     2|   4600|
|  Computers|Southeast|     1|   1200|
|Peripherals|Southeast|     3|    975|
|      Audio|    North|     1|    500|
|Peripherals|    North|     2|    435|
|    Storage|Southeast|     1|    400|
|Peripherals|    South|     1|    320|
+-----------+---------+------+-------+
only showing top 8 rows


In [ ]:
# Window Functions
# Compute values across a group WITHOUT collapsing the DataFrame.
# Equivalent to Pandas .groupby().transform()

# 🐼 Pandas
pdf2["avg_price_cat"] = pdf2.groupby("category")["price"].transform("mean").round(2)
pdf2["rank_in_cat"]   = pdf2.groupby("category")["price"].rank(ascending=False)

print("🐼 Pandas transform + rank:")
display(
    pdf2[["product", "category", "price", "avg_price_cat", "rank_in_cat"]]
    .sort_values(["category", "rank_in_cat"])
)

# 🔥 Spark — Window specs
w_cat       = Window.partitionBy("category")
w_cat_price = Window.partitionBy("category").orderBy(F.col("price").desc())

sdf_win = (
    sdf2
    .withColumn("avg_price_cat", F.round(F.avg("price").over(w_cat), 2))
    .withColumn("rank_in_cat",   F.rank().over(w_cat_price))
)
print("\n🔥 Spark Window functions:")
sdf_win.select("product", "category", "price", "avg_price_cat", "rank_in_cat") \
       .orderBy("category", "rank_in_cat") \
       .show()

🐼 Pandas transform + rank:


,product,category,price,avg_price_cat,rank_in_cat
14,Headset,Audio,280,265.00,1.0
4,Headset,Audio,250,265.00,2.0
8,USB Cable,Cables,25,25.00,1.0
0,Notebook,Computers,3500,2250.00,1.0
10,Notebook,Computers,3200,2250.00,2.0
3,Monitor,Computers,1200,2250.00,3.0
11,Monitor,Computers,1100,2250.00,4.0
5,Webcam,Peripherals,320,151.67,1.0
2,Keyboard,Peripherals,150,151.67,2.0
12,Keyboard,Peripherals,145,151.67,3.0



🔥 Spark Window functions:
+-----------+-----------+-----+-------------+-----------+
|    product|   category|price|avg_price_cat|rank_in_cat|
+-----------+-----------+-----+-------------+-----------+
|    Headset|      Audio|  280|        265.0|          1|
|    Headset|      Audio|  250|        265.0|          2|
|  USB Cable|     Cables|   25|         25.0|          1|
|   Notebook|  Computers| 3500|       2250.0|          1|
|   Notebook|  Computers| 3200|       2250.0|          2|
|    Monitor|  Computers| 1200|       2250.0|          3|
|    Monitor|  Computers| 1100|       2250.0|          4|
|     Webcam|Peripherals|  320|       151.67|          1|
|   Keyboard|Peripherals|  150|       151.67|          2|
|   Keyboard|Peripherals|  145|       151.67|          3|
|    USB Hub|Peripherals|  120|       151.67|          4|
|      Mouse|Peripherals|   90|       151.67|          5|
|      Mouse|Peripherals|   85|       151.67|          6|
|External HD|    Storage|  400|        230.0|

---
## 7. 🔗 Joins

| Type | Pandas | Spark | Behavior |
|------|--------|-------|----------|
| Inner | `how="inner"` | `how="inner"` | Only matching rows |
| Left | `how="left"` | `how="left"` | All left + matched right |
| Right | `how="right"` | `how="right"` | All right + matched left |
| Full outer | `how="outer"` | `how="outer"` | All rows from both |
| Stack rows | `pd.concat([...])` | `df1.union(df2)` | Append rows (UNION) |

In [ ]:
# Reference / lookup table
suppliers_data = {
    "category":     ["Computers", "Peripherals", "Audio", "Storage", "Cables"],
    "supplier":     ["TechPro", "OfficeMax", "SoundWave", "DataStore", "ConnectHub"],
    "lead_days":    [7, 3, 5, 4, 2],
    "discount_pct": [0.05, 0.10, 0.08, 0.12, 0.15]
}
pdf_sup = pd.DataFrame(suppliers_data)
sdf_sup = spark.createDataFrame(pdf_sup)
display(pdf_sup)

,category,supplier,lead_days,discount_pct
0,Computers,TechPro,7,0.05
1,Peripherals,OfficeMax,3,0.10
2,Audio,SoundWave,5,0.08
3,Storage,DataStore,4,0.12
4,Cables,ConnectHub,2,0.15


In [ ]:
# Inner Join
pd_joined = pdf2.merge(pdf_sup, on="category", how="inner")
print(f"🐼 Pandas inner join — shape: {pd_joined.shape}")
display(pd_joined[["product", "category", "price", "supplier", "lead_days"]].head(5))

sp_joined = sdf2.join(sdf_sup, on="category", how="inner")
print(f"\n🔥 Spark inner join — count: {sp_joined.count()}")
sp_joined.select("product", "category", "price", "supplier", "lead_days").show(5)

🐼 Pandas inner join — shape: (15, 24)


,product,category,price,supplier,lead_days
0,Notebook,Computers,3500,TechPro,7
1,Mouse,Peripherals,85,OfficeMax,3
2,Keyboard,Peripherals,150,OfficeMax,3
3,Monitor,Computers,1200,TechPro,7
4,Headset,Audio,250,SoundWave,5



🔥 Spark inner join — count: 15
+--------+-----------+-----+---------+---------+
| product|   category|price| supplier|lead_days|
+--------+-----------+-----+---------+---------+
| Monitor|  Computers| 1100|  TechPro|        7|
|Notebook|  Computers| 3200|  TechPro|        7|
| Monitor|  Computers| 1200|  TechPro|        7|
|Notebook|  Computers| 3500|  TechPro|        7|
|   Mouse|Peripherals|   90|OfficeMax|        3|
+--------+-----------+-----+---------+---------+
only showing top 5 rows


In [ ]:
# Left Join + derived column using the discount
pd_left = pdf2.merge(pdf_sup, on="category", how="left")
pd_left["final_price"] = (pd_left["price"] * (1 - pd_left["discount_pct"].fillna(0))).round(2)
print("🐼 Pandas left join + final_price:")
display(pd_left[["product", "price", "discount_pct", "final_price"]].head(6))

sp_left = (
    sdf2
    .join(sdf_sup, on="category", how="left")
    .withColumn(
        "final_price",
        F.round(F.col("price") * (1 - F.coalesce(F.col("discount_pct"), F.lit(0))), 2)
    )
)
print("\n🔥 Spark left join + final_price:")
sp_left.select("product", "price", "discount_pct", "final_price").show(6)

🐼 Pandas left join + final_price:


,product,price,discount_pct,final_price
0,Notebook,3500,0.05,3325.0
1,Mouse,85,0.10,76.5
2,Keyboard,150,0.10,135.0
3,Monitor,1200,0.05,1140.0
4,Headset,250,0.08,230.0
5,Webcam,320,0.10,288.0



🔥 Spark left join + final_price:
+--------+-----+------------+-----------+
| product|price|discount_pct|final_price|
+--------+-----+------------+-----------+
|Notebook| 3500|        0.05|     3325.0|
| Monitor| 1200|        0.05|     1140.0|
|   Mouse|   85|         0.1|       76.5|
|Keyboard|  150|         0.1|      135.0|
| Headset|  250|        0.08|      230.0|
|  Webcam|  320|         0.1|      288.0|
+--------+-----+------------+-----------+
only showing top 6 rows


In [ ]:
# Union — stacking DataFrames with the same schema
sdf_north = sdf.filter(F.col("region") == "North")
sdf_south = sdf.filter(F.col("region") == "South")

pd_union = pd.concat(
    [pdf[pdf["region"]=="North"], pdf[pdf["region"]=="South"]],
    ignore_index=True
)
print(f"🐼 pd.concat    — shape : {pd_union.shape}")
display(pd_union)

sp_union = sdf_north.union(sdf_south)
print(f"🔥 Spark .union — count : {sp_union.count()}")
sp_union.show()

🐼 pd.concat    — shape : (9, 8)


,order_id,product,category,price,quantity,rating,region,sale_date
0,1002,Mouse,Peripherals,85,3,4.2,North,2024-01-18
1,1005,Headset,Audio,250,2,3.9,North,2024-02-14
2,1008,Pendrive,Storage,60,5,4.0,North,2024-03-22
3,1011,Notebook,Computers,3200,2,4.9,North,2024-05-02
4,1014,Mouse,Peripherals,90,2,4.1,North,2024-06-15
5,1001,Notebook,Computers,3500,1,4.8,South,2024-01-15
6,1006,Webcam,Peripherals,320,1,4.1,South,2024-03-01
7,1009,USB Cable,Cables,25,10,NaN,South,2024-04-05
8,1012,Monitor,Computers,1100,1,4.6,South,2024-05-20


🔥 Spark .union — count : 9
+--------+---------+-----------+-----+--------+------+------+----------+
|order_id|  product|   category|price|quantity|rating|region| sale_date|
+--------+---------+-----------+-----+--------+------+------+----------+
|    1002|    Mouse|Peripherals|   85|       3|   4.2| North|2024-01-18|
|    1005|  Headset|      Audio|  250|       2|   3.9| North|2024-02-14|
|    1008| Pendrive|    Storage|   60|       5|   4.0| North|2024-03-22|
|    1011| Notebook|  Computers| 3200|       2|   4.9| North|2024-05-02|
|    1014|    Mouse|Peripherals|   90|       2|   4.1| North|2024-06-15|
|    1001| Notebook|  Computers| 3500|       1|   4.8| South|2024-01-15|
|    1006|   Webcam|Peripherals|  320|       1|   4.1| South|2024-03-01|
|    1009|USB Cable|     Cables|   25|      10|   NaN| South|2024-04-05|
|    1012|  Monitor|  Computers| 1100|       1|   4.6| South|2024-05-20|
+--------+---------+-----------+-----+--------+------+------+----------+



---
## 8. 🗄️ Spark SQL

> Spark SQL lets you query **any** Spark DataFrame using standard SQL.  
> Register a DataFrame as a **temporary view** and write SQL against it —  
> ideal for analysts who already know SQL, and it uses the same optimized query planner.

In [ ]:
# Register views — they live for the duration of the SparkSession
sdf2.createOrReplaceTempView("sales")
sdf_sup.createOrReplaceTempView("suppliers")
print("Views registered: sales, suppliers")

Views registered: sales, suppliers


In [ ]:
# Simple SELECT with WHERE and ORDER BY
spark.sql("""
    SELECT  product, category, price, quantity, total
    FROM    sales
    WHERE   price > 100
    ORDER   BY total DESC
    LIMIT   5
""").show()

+--------+---------+-----+--------+-----+
| product| category|price|quantity|total|
+--------+---------+-----+--------+-----+
|Notebook|Computers| 3200|       2| 6400|
|Notebook|Computers| 3500|       1| 3500|
| Monitor|Computers| 1200|       1| 1200|
| Monitor|Computers| 1100|       1| 1100|
| Headset|    Audio|  250|       2|  500|
+--------+---------+-----+--------+-----+



In [ ]:
# GROUP BY with multiple aggregations
spark.sql("""
    SELECT
        category,
        COUNT(*)               AS orders,
        SUM(total)             AS total_revenue,
        ROUND(AVG(price), 2)   AS avg_price,
        ROUND(AVG(rating), 2)  AS avg_rating,
        MAX(price)             AS max_price
    FROM  sales
    GROUP BY category
    ORDER BY total_revenue DESC
""").show()

+-----------+------+-------------+---------+----------+---------+
|   category|orders|total_revenue|avg_price|avg_rating|max_price|
+-----------+------+-------------+---------+----------+---------+
|  Computers|     4|        12200|   2250.0|      4.75|     3500|
|Peripherals|     6|         1730|   151.67|      4.27|      320|
|      Audio|     2|          780|    265.0|      3.95|      280|
|    Storage|     2|          700|    230.0|       4.3|      400|
|     Cables|     1|          250|     25.0|       NaN|       25|
+-----------+------+-------------+---------+----------+---------+



In [ ]:
# JOIN in SQL
spark.sql("""
    SELECT
        s.product,
        s.category,
        s.price,
        sup.supplier,
        sup.lead_days,
        ROUND(s.price * (1 - COALESCE(sup.discount_pct, 0)), 2) AS final_price
    FROM   sales s
    LEFT   JOIN suppliers sup ON s.category = sup.category
    ORDER  BY s.price DESC
    LIMIT  8
""").show(truncate=False)

+-----------+-----------+-----+---------+---------+-----------+
|product    |category   |price|supplier |lead_days|final_price|
+-----------+-----------+-----+---------+---------+-----------+
|Notebook   |Computers  |3500 |TechPro  |7        |3325.0     |
|Notebook   |Computers  |3200 |TechPro  |7        |3040.0     |
|Monitor    |Computers  |1200 |TechPro  |7        |1140.0     |
|Monitor    |Computers  |1100 |TechPro  |7        |1045.0     |
|External HD|Storage    |400  |DataStore|4        |352.0      |
|Webcam     |Peripherals|320  |OfficeMax|3        |288.0      |
|Headset    |Audio      |280  |SoundWave|5        |257.6      |
|Headset    |Audio      |250  |SoundWave|5        |230.0      |
+-----------+-----------+-----+---------+---------+-----------+



In [ ]:
# Window Function in SQL
spark.sql("""
    SELECT
        product,
        category,
        price,
        ROUND(AVG(price) OVER (PARTITION BY category), 2)          AS avg_cat_price,
        RANK()           OVER (PARTITION BY category ORDER BY price DESC) AS rank_in_cat
    FROM  sales
    ORDER BY category, rank_in_cat
""").show()

+-----------+-----------+-----+-------------+-----------+
|    product|   category|price|avg_cat_price|rank_in_cat|
+-----------+-----------+-----+-------------+-----------+
|    Headset|      Audio|  280|        265.0|          1|
|    Headset|      Audio|  250|        265.0|          2|
|  USB Cable|     Cables|   25|         25.0|          1|
|   Notebook|  Computers| 3500|       2250.0|          1|
|   Notebook|  Computers| 3200|       2250.0|          2|
|    Monitor|  Computers| 1200|       2250.0|          3|
|    Monitor|  Computers| 1100|       2250.0|          4|
|     Webcam|Peripherals|  320|       151.67|          1|
|   Keyboard|Peripherals|  150|       151.67|          2|
|   Keyboard|Peripherals|  145|       151.67|          3|
|    USB Hub|Peripherals|  120|       151.67|          4|
|      Mouse|Peripherals|   90|       151.67|          5|
|      Mouse|Peripherals|   85|       151.67|          6|
|External HD|    Storage|  400|        230.0|          1|
|   Pendrive| 

---
## 9. ⚡ Lazy Evaluation & Performance

### Transformations vs Actions

This is the **most fundamental** difference between Spark and Pandas.

| | Type | Examples | Effect |
|-|------|---------|--------|
| **Transformations** | Lazy | `filter`, `select`, `withColumn`, `groupBy`, `join` | Builds a plan — **nothing runs** |
| **Actions** | Eager | `show()`, `count()`, `collect()`, `write`, `toPandas()` | **Executes the full plan** |

```
  TRANSFORMATION ──▶ TRANSFORMATION ──▶ TRANSFORMATION ──▶  ACTION
    (lazy, free)       (lazy, free)       (lazy, free)      (runs!)
    .filter()          .withColumn()      .groupBy()        .show()
```

Spark uses this to build an **optimized query plan** before executing — similar to how a SQL engine works internally.

In [ ]:
import time

# Building the plan — INSTANT, no data is read or processed yet
t0 = time.time()
plan = (
    sdf
    .filter(F.col("price") > 100)                              # lazy
    .withColumn("total", F.col("price") * F.col("quantity"))   # lazy
    .groupBy("category")                                       # lazy
    .agg(F.sum("total").alias("revenue"))                      # lazy
    .orderBy(F.col("revenue").desc())                          # lazy
)
print(f"Building the plan : {(time.time()-t0)*1000:.1f} ms  ← No execution yet!")

# The ACTION — this is where Spark actually runs all steps above
t0 = time.time()
plan.show()
print(f"Executing (action): {(time.time()-t0)*1000:.1f} ms  ← All work happens here")

Building the plan : 37.4 ms  ← No execution yet!
+-----------+-------+
|   category|revenue|
+-----------+-------+
|  Computers|  12200|
|Peripherals|   1295|
|      Audio|    780|
|    Storage|    400|
+-----------+-------+

Executing (action): 322.0 ms  ← All work happens here


In [ ]:
# .explain() shows the logical and physical plan Spark will execute
# Useful for understanding and debugging performance
plan.explain(mode="simple")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [revenue#5404L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(revenue#5404L DESC NULLS LAST, 4), ENSURE_REQUIREMENTS, [plan_id=6155]
      +- HashAggregate(keys=[category#4428], functions=[sum(total#5403L)])
         +- Exchange hashpartitioning(category#4428, 4), ENSURE_REQUIREMENTS, [plan_id=6152]
            +- HashAggregate(keys=[category#4428], functions=[partial_sum(total#5403L)])
               +- Project [category#4428, (price#4429L * quantity#4430L) AS total#5403L]
                  +- Filter (isnotnull(price#4429L) AND (price#4429L > 100))
                     +- Scan ExistingRDD[order_id#4426L,product#4427,category#4428,price#4429L,quantity#4430L,rating#4431,region#4432,sale_date#4433]




In [ ]:
# Caching
# .cache() stores the DataFrame in memory after the first action.
# Subsequent actions reuse it instead of recomputing from scratch.

sdf_cached = sdf2.cache()

t0 = time.time()
_ = sdf_cached.count()   # first action — computes AND caches
print(f"First  action (compute + cache): {(time.time()-t0)*1000:.1f} ms")

t0 = time.time()
_ = sdf_cached.count()   # second action — reads from memory
print(f"Second action (from cache):      {(time.time()-t0)*1000:.1f} ms  ← Faster!")

sdf_cached.unpersist()
print("Cache released.")

First  action (compute + cache): 542.6 ms
Second action (from cache):      92.0 ms  ← Faster!
Cache released.


In [ ]:
# Partitions — the unit of parallelism in Spark
print(f"Default partitions          : {sdf.rdd.getNumPartitions()}")

# repartition — full shuffle, evenly distributes data (use before heavy joins/aggregations)
sdf_rep = sdf.repartition(4, "category")
print(f"After repartition(4, 'cat') : {sdf_rep.rdd.getNumPartitions()}")

# coalesce — reduces partitions WITHOUT a full shuffle (faster, but less balanced)
sdf_coal = sdf.coalesce(2)
print(f"After coalesce(2)           : {sdf_coal.rdd.getNumPartitions()}")

Default partitions          : 2
After repartition(4, 'cat') : 4
After coalesce(2)           : 2


---
## 10. Examples: Pandas & Spark

In [ ]:
# Clean DataFrames for the exercises
pdf_ex = pd.DataFrame(data)
sdf_ex = spark.createDataFrame(pdf_ex)
sdf_ex.createOrReplaceTempView("sales_ex")

In [ ]:
# ── Solution 1 ────────────────────────────────────────────────────────────────
# Find all products where rating >= 4.5 OR price < 100.
# Show: product, category, price, rating — sorted by rating descending.
# Ignore rows where rating is null.
print("🐼 Pandas:")
display(
    pdf_ex[
        (pdf_ex["rating"] >= 4.5) | (pdf_ex["price"] < 100)
    ]
    .dropna(subset=["rating"])
    [["product", "category", "price", "rating"]]
    .sort_values("rating", ascending=False)
    .reset_index(drop=True)
)

print("\n🔥 Spark:")
(
    sdf_ex
    .filter((F.col("rating") >= 4.5) | (F.col("price") < 100))
    .dropna(subset=["rating"])
    .select("product", "category", "price", "rating")
    .orderBy(F.col("rating").desc())
).show()

🐼 Pandas:


,product,category,price,rating
0,Notebook,Computers,3200,4.9
1,Notebook,Computers,3500,4.8
2,Monitor,Computers,1200,4.7
3,Monitor,Computers,1100,4.6
4,External HD,Storage,400,4.6
5,Keyboard,Peripherals,150,4.5
6,Mouse,Peripherals,85,4.2
7,Mouse,Peripherals,90,4.1
8,Pendrive,Storage,60,4.0



🔥 Spark:
+-----------+-----------+-----+------+
|    product|   category|price|rating|
+-----------+-----------+-----+------+
|   Notebook|  Computers| 3200|   4.9|
|   Notebook|  Computers| 3500|   4.8|
|    Monitor|  Computers| 1200|   4.7|
|    Monitor|  Computers| 1100|   4.6|
|External HD|    Storage|  400|   4.6|
|   Keyboard|Peripherals|  150|   4.5|
|      Mouse|Peripherals|   85|   4.2|
|      Mouse|Peripherals|   90|   4.1|
|   Pendrive|    Storage|   60|   4.0|
+-----------+-----------+-----+------+



In [ ]:
# ── Solution 2 ────────────────────────────────────────────────────────────────
# Calculate total revenue (price × quantity) per region.
# Sort by revenue descending.
# Also compute each region's percentage share of total revenue (round to 1 decimal).
print("🐼 Pandas:")
pdf_ex2 = pdf_ex.copy()
pdf_ex2["revenue"] = pdf_ex2["price"] * pdf_ex2["quantity"]
pd_reg = (
    pdf_ex2
    .groupby("region")
    .agg(total_revenue=("revenue", "sum"))
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
pd_reg["pct_share"] = (pd_reg["total_revenue"] / pd_reg["total_revenue"].sum() * 100).round(1)
display(pd_reg)

print("\n🔥 Spark:")
total_rev = sdf_ex.select(F.sum(F.col("price") * F.col("quantity"))).collect()[0][0]
(
    sdf_ex
    .withColumn("revenue", F.col("price") * F.col("quantity"))
    .groupBy("region")
    .agg(F.sum("revenue").alias("total_revenue"))
    .withColumn("pct_share", F.round(F.col("total_revenue") / total_rev * 100, 1))
    .orderBy(F.col("total_revenue").desc())
).show()

🐼 Pandas:


,region,total_revenue,pct_share
0,North,7635,48.8
1,South,5170,33.0
2,Southeast,2855,18.2



🔥 Spark:
+---------+-------------+---------+
|   region|total_revenue|pct_share|
+---------+-------------+---------+
|    North|         7635|     48.8|
|    South|         5170|     33.0|
|Southeast|         2855|     18.2|
+---------+-------------+---------+



In [ ]:
# ── Solution 3 ────────────────────────────────────────────────────────────────
# Join the sales data with the suppliers table.
# Create a "final_price" column applying each supplier's discount.
# Return only orders where lead_days <= 4, sorted by final_price descending.

pdf_sup_ex = pd.DataFrame(suppliers_data)
sdf_sup_ex = spark.createDataFrame(pdf_sup_ex)

print("🐼 Pandas:")
display(
    pdf_ex
    .merge(pdf_sup_ex, on="category", how="left")
    .assign(final_price=lambda d: (d["price"] * (1 - d["discount_pct"].fillna(0))).round(2))
    .query("lead_days <= 4")
    [["product", "category", "price", "supplier", "lead_days", "final_price"]]
    .sort_values("final_price", ascending=False)
    .reset_index(drop=True)
)

print("\n🔥 Spark:")
(
    sdf_ex
    .join(sdf_sup_ex, on="category", how="left")
    .withColumn(
        "final_price",
        F.round(F.col("price") * (1 - F.coalesce(F.col("discount_pct"), F.lit(0))), 2)
    )
    .filter(F.col("lead_days") <= 4)
    .select("product", "category", "price", "supplier", "lead_days", "final_price")
    .orderBy(F.col("final_price").desc())
).show()

🐼 Pandas:


,product,category,price,supplier,lead_days,final_price
0,External HD,Storage,400,DataStore,4,352.00
1,Webcam,Peripherals,320,OfficeMax,3,288.00
2,Keyboard,Peripherals,150,OfficeMax,3,135.00
3,Keyboard,Peripherals,145,OfficeMax,3,130.50
4,USB Hub,Peripherals,120,OfficeMax,3,108.00
5,Mouse,Peripherals,90,OfficeMax,3,81.00
6,Mouse,Peripherals,85,OfficeMax,3,76.50
7,Pendrive,Storage,60,DataStore,4,52.80
8,USB Cable,Cables,25,ConnectHub,2,21.25



🔥 Spark:
+-----------+-----------+-----+----------+---------+-----------+
|    product|   category|price|  supplier|lead_days|final_price|
+-----------+-----------+-----+----------+---------+-----------+
|External HD|    Storage|  400| DataStore|        4|      352.0|
|     Webcam|Peripherals|  320| OfficeMax|        3|      288.0|
|   Keyboard|Peripherals|  150| OfficeMax|        3|      135.0|
|   Keyboard|Peripherals|  145| OfficeMax|        3|      130.5|
|    USB Hub|Peripherals|  120| OfficeMax|        3|      108.0|
|      Mouse|Peripherals|   90| OfficeMax|        3|       81.0|
|      Mouse|Peripherals|   85| OfficeMax|        3|       76.5|
|   Pendrive|    Storage|   60| DataStore|        4|       52.8|
|  USB Cable|     Cables|   25|ConnectHub|        2|      21.25|
+-----------+-----------+-----+----------+---------+-----------+



In [ ]:
# ── Solution 4 — SQL ──────────────────────────────────────────────────────────
# Using ONLY Spark SQL, write a query that:
#   1. Calculates total revenue (price × quantity) per category
#   2. Ranks the categories by revenue using a window function
#   3. Returns only the top 3 categories
spark.sql("""
    WITH ranked AS (
        SELECT
            category,
            SUM(price * quantity) AS total_revenue,
            RANK() OVER (ORDER BY SUM(price * quantity) DESC) AS revenue_rank
        FROM   sales_ex
        GROUP  BY category
    )
    SELECT  category, total_revenue, revenue_rank
    FROM    ranked
    WHERE   revenue_rank <= 3
    ORDER   BY revenue_rank
""").show()

+-----------+-------------+------------+
|   category|total_revenue|revenue_rank|
+-----------+-------------+------------+
|  Computers|        12200|           1|
|Peripherals|         1730|           2|
|      Audio|          780|           3|
+-----------+-------------+------------+



---
## 📋 Full Reference: Pandas vs PySpark

| Operation | 🐼 Pandas | 🔥 PySpark |
|-----------|-----------|------------|
| Create from dict | `pd.DataFrame(d)` | `spark.createDataFrame(pdf)` |
| Read CSV | `pd.read_csv("f.csv")` | `spark.read.csv("f.csv", header=True)` |
| Read Parquet | `pd.read_parquet("f.pq")` | `spark.read.parquet("f.pq")` |
| Show rows | `df.head(5)` | `df.show(5)` |
| Schema / types | `df.dtypes` | `df.printSchema()` |
| Dimensions | `df.shape` | `(df.count(), len(df.columns))` |
| Select columns | `df[["a", "b"]]` | `df.select("a", "b")` |
| Filter rows | `df[df.col > x]` | `df.filter(df.col > x)` |
| SQL filter | `df.query("col > x")` | `df.filter("col > x")` |
| Add column | `df["c"] = expr` | `df.withColumn("c", expr)` |
| Rename column | `df.rename(columns={"a": "b"})` | `df.withColumnRenamed("a", "b")` |
| Drop column | `df.drop(columns=["c"])` | `df.drop("c")` |
| Fill nulls | `df.fillna({"c": v})` | `df.fillna({"c": v})` |
| Drop null rows | `df.dropna()` | `df.dropna()` |
| Custom function | `df["c"].apply(fn)` | `df.withColumn("c", F.udf(fn)(col))` |
| If-else column | `np.where(cond, a, b)` | `F.when(cond, a).otherwise(b)` |
| Sort | `df.sort_values("c")` | `df.orderBy("c")` |
| GroupBy | `df.groupby("c").agg(...)` | `df.groupBy("c").agg(...)` |
| Sum | `("c", "sum")` | `F.sum("c")` |
| Average | `("c", "mean")` | `F.avg("c")` |
| Count | `("c", "count")` | `F.count("c")` |
| Group transform | `.groupby().transform()` | `F.avg().over(Window.partitionBy(...))` |
| Join | `df.merge(df2, on="k", how="left")` | `df.join(df2, on="k", how="left")` |
| Stack rows | `pd.concat([df1, df2])` | `df1.union(df2)` |
| Run SQL | — | `spark.sql("SELECT ...")` |
| Count rows | `len(df)` | `df.count()` ← action! |
| To Pandas | — | `df.toPandas()` ← action! |
| Write Parquet | `df.to_parquet("f.pq")` | `df.write.parquet("f.pq")` |

---

## 🚀 What's Next

```
✅ Pandas   → DataFrames, transforms, groupby, joins
✅ PySpark  → Same concepts, distributed, lazy evaluation, SQL, window functions

⏭️  Next sessions:
    → Delta Lake      — ACID transactions on data lakes
    → Spark Streaming — real-time data processing
    → MLlib           — distributed machine learning
    → Spark on Cloud  — Databricks, AWS EMR, GCP Dataproc
    → Performance     — broadcast joins, AQE, partition strategies
```

In [ ]:
# Always stop the session when done — releases all resources
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.
